Snowflake Integrations - Complete Notes (Azure Focus)
*Co-authored with CoCo*

# Snowflake Integration Types — Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                    SNOWFLAKE INTEGRATIONS                        │
├──────────────┬──────────────┬───────────────┬───────────────────┤
│   STORAGE    │ NOTIFICATION │      API      │     SECURITY      │
├──────────────┼──────────────┼───────────────┼───────────────────┤
│ Azure Blob   │ Event Grid   │ Azure Functions│ Azure AD (SSO)   │
│ ADLS Gen2    │ Storage Queue│ API Management │ OAuth 2.0        │
│              │              │ Any REST API   │ SCIM             │
├──────────────┼──────────────┼───────────────┼───────────────────┤
│ Reads/writes │ Receives or  │ Calls external │ Authenticates    │
│ files in     │ sends event  │ HTTPS endpoints│ users/apps via   │
│ blob storage │ notifications│ from SQL       │ federated auth   │
└──────────────┴──────────────┴───────────────┴───────────────────┘
```

There is also a **Catalog Integration** (used for Apache Iceberg tables with external catalogs like AWS Glue or Polaris) and an **External Access Integration** (used by UDFs/stored procedures to call external network endpoints), but these are less common for general data pipeline use.

# 1. Storage Integration (Azure)

## What It Does
Allows Snowflake to securely read/write data in **Azure Blob Storage** or **Azure Data Lake Storage Gen2** without passing credentials in every SQL command. Instead, a one-time trust relationship is established between Snowflake and your Azure tenant.

## Azure Components Involved
- **Azure Storage Account** (Blob or ADLS Gen2)
- **Azure AD App Registration** (Service Principal created by Snowflake)
- **Tenant ID** from Azure AD (found under Azure AD > Properties)

## How It Works (Flow)
```
1. You create a Storage Integration in Snowflake (specifying tenant ID and allowed locations)
2. Snowflake generates an Azure AD Service Principal in its own Azure tenant
3. You open the consent URL (from DESC INTEGRATION output) to grant Snowflake's app access
4. You assign an RBAC role (e.g., Storage Blob Data Contributor) to the Service Principal on your Storage Account
5. Snowflake uses this trust relationship to read/write blobs directly — no SAS tokens or keys needed
```

## When You Need It
- Loading data from Azure Blob into Snowflake tables (`COPY INTO <table>`)
- Unloading data from Snowflake to Azure Blob (`COPY INTO @<stage>`)
- Creating external stages pointing to Azure containers
- Snowpipe (automated data ingestion from blob storage)
- External tables backed by files in blob storage

In [ ]:
%%sql -r create_storage_int
-- ============================================================
-- STORAGE INTEGRATION: Azure Blob Storage
-- ============================================================

-- Step 1: Create the Storage Integration
CREATE OR REPLACE STORAGE INTEGRATION azure_blob_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'AZURE'
  ENABLED = TRUE
  AZURE_TENANT_ID = '<your-azure-tenant-id>'  -- from Azure AD > Properties > Tenant ID
  STORAGE_ALLOWED_LOCATIONS = (
    'azure://myaccount.blob.core.windows.net/mycontainer/path/'
  )
  STORAGE_BLOCKED_LOCATIONS = (
    'azure://myaccount.blob.core.windows.net/mycontainer/sensitive/'
  );

In [ ]:
%%sql -r desc_storage_int
-- Step 2: Retrieve the Service Principal details Snowflake generated
-- You need these to grant access in Azure
DESC INTEGRATION azure_blob_int;

-- Key output columns:
-- AZURE_CONSENT_URL       → Open this URL to grant consent to Snowflake's app
-- AZURE_MULTI_TENANT_APP_NAME → The app name Snowflake registered

## Azure Side Setup (After Creating Integration)

### Step-by-step in Azure Portal:

1. **Open the Consent URL** from the `DESC INTEGRATION` output → Accept the permissions prompt to authorize Snowflake's app in your tenant
2. **Go to Azure Portal → Storage Account → Access Control (IAM)**
3. **Add Role Assignment:**
   - Role: `Storage Blob Data Contributor` (for read + write) or `Storage Blob Data Reader` (for read-only)
   - Assign to: The Snowflake service principal (search by the `AZURE_MULTI_TENANT_APP_NAME` value from the DESC output)
4. **Verify** access by running a `LIST @<stage_name>` command in Snowflake

> **Note:** The consent URL expires after a short time. Complete this step promptly after running `DESC INTEGRATION`.

```
┌────────────────────────────────────────────────────────┐
│                   TRUST FLOW                            │
│                                                        │
│  Snowflake Account                                     │
│       │                                                │
│       │ (authenticates as Service Principal)            │
│       ▼                                                │
│  Azure AD ──── validates identity ────► Storage Account│
│       │                                    │           │
│       └── RBAC: Blob Data Contributor ─────┘           │
└────────────────────────────────────────────────────────┘
```

In [ ]:
%%sql -r create_stage
-- Step 3: Create an External Stage using the Storage Integration
CREATE OR REPLACE STAGE my_azure_stage
  STORAGE_INTEGRATION = azure_blob_int
  URL = 'azure://myaccount.blob.core.windows.net/mycontainer/data/'
  FILE_FORMAT = (TYPE = 'CSV' FIELD_DELIMITER = ',' SKIP_HEADER = 1);

-- Step 4: Verify access by listing files
-- LIST @my_azure_stage;

In [ ]:
%%sql -r copy_examples
-- Step 5: Load data from Azure Blob into a Snowflake table
-- COPY INTO my_table
--   FROM @my_azure_stage
--   FILE_FORMAT = (TYPE = 'CSV')
--   ON_ERROR = 'CONTINUE';

-- Step 6: Unload data from Snowflake to Azure Blob
-- COPY INTO @my_azure_stage/export/
--   FROM my_table
--   FILE_FORMAT = (TYPE = 'PARQUET')
--   HEADER = TRUE;

# 2. Notification Integration (Azure)

## What It Does
Enables Snowflake to **receive** event notifications (inbound) or **send** notifications (outbound) via Azure messaging services. This is the mechanism that triggers automated actions in Snowflake when something happens in Azure, or vice versa.

## Azure Components Involved
- **Azure Event Grid** (publishes events when blobs are created/modified)
- **Azure Storage Queue** (holds event messages for Snowflake to consume)
- **Azure AD Tenant ID** (for authentication)

## Two Directions

| Direction | Use Case | Azure Service | Snowflake Reads/Writes |
|---|---|---|---|
| **Inbound** | Snowpipe auto-ingest ("new file arrived → load it") | Storage Queue | Snowflake reads from the queue |
| **Outbound** | Send alerts/errors from Snowflake to Azure | Event Grid Topic | Snowflake publishes to the topic |

## When You Need It
- **Snowpipe auto-ingest**: Automatically load new files as they land in blob storage (requires inbound notification)
- **Error notifications**: Get alerted when Snowpipe or Tasks fail (outbound)
- **Custom notifications**: Send messages from stored procedures to Azure Event Grid for downstream processing

## Important Distinction
- **Inbound** notifications use `AZURE_STORAGE_QUEUE` as the provider
- **Outbound** notifications use `AZURE_EVENT_GRID` as the provider and require the `DIRECTION = OUTBOUND` parameter

In [ ]:
%%sql -r create_notify_inbound
-- ============================================================
-- NOTIFICATION INTEGRATION: Inbound (Azure Storage Queue)
-- Used for Snowpipe auto-ingest
-- ============================================================

CREATE OR REPLACE NOTIFICATION INTEGRATION azure_notify_inbound
  ENABLED = TRUE
  TYPE = QUEUE
  NOTIFICATION_PROVIDER = AZURE_STORAGE_QUEUE
  AZURE_STORAGE_QUEUE_PRIMARY_URI = 'https://mystorageacct.queue.core.windows.net/snowpipe-queue'
  AZURE_TENANT_ID = '<your-azure-tenant-id>';

In [ ]:
%%sql -r create_notify_outbound
-- ============================================================
-- NOTIFICATION INTEGRATION: Outbound (Azure Event Grid)
-- Used for sending alerts from Snowflake
-- ============================================================

CREATE OR REPLACE NOTIFICATION INTEGRATION azure_notify_outbound
  ENABLED = TRUE
  TYPE = QUEUE
  DIRECTION = OUTBOUND
  NOTIFICATION_PROVIDER = AZURE_EVENT_GRID
  AZURE_EVENT_GRID_TOPIC_ENDPOINT = 'https://mytopic.eastus-1.eventgrid.azure.net/api/events'
  AZURE_TENANT_ID = '<your-azure-tenant-id>';

## Azure Side Setup for Notification Integration

### For Inbound (Snowpipe auto-ingest):
1. **Create an Azure Storage Queue** in the same storage account as your blobs
2. **Create an Event Grid Subscription** on the blob container:
   - Event types: `Blob Created` (and optionally `Blob Renamed`)
   - Endpoint type: `Storage Queue`
   - Endpoint: the queue you created in step 1
3. **Grant Snowflake access** to read the queue — open the consent URL from `DESC INTEGRATION` output and accept
4. **Assign the role** `Storage Queue Data Reader` to the Snowflake service principal on the queue's storage account
5. **Create a Pipe** in Snowflake with `AUTO_INGEST = TRUE` and reference the notification integration

### For Outbound (Alerts):
1. **Create an Event Grid Topic** in Azure
2. **Grant Snowflake access** to publish to the topic — open the consent URL and assign the `EventGrid Data Sender` role
3. **Subscribe** your downstream services (Logic App, Azure Function, etc.) to the Event Grid Topic

```
┌──────────────────────────────────────────────────────────────┐
│            INBOUND NOTIFICATION FLOW                          │
│                                                              │
│  Azure Blob Storage                                          │
│       │ (new file uploaded)                                  │
│       ▼                                                      │
│  Event Grid Subscription (triggers on BlobCreated)           │
│       │                                                      │
│       ▼                                                      │
│  Azure Storage Queue (holds the event message)               │
│       │                                                      │
│       ▼                                                      │
│  Snowflake Notification Integration (polls the queue)        │
│       │                                                      │
│       ▼                                                      │
│  Snowpipe → COPY INTO → loads data into target table         │
└──────────────────────────────────────────────────────────────┘
```

In [ ]:
%%sql -r snowpipe_example
-- ============================================================
-- SNOWPIPE with Auto-Ingest using Notification Integration
-- ============================================================

-- CREATE PIPE my_auto_pipe
--   AUTO_INGEST = TRUE
--   INTEGRATION = 'azure_notify_inbound'
--   AS
--     COPY INTO my_target_table
--       FROM @my_azure_stage
--       FILE_FORMAT = (TYPE = 'JSON');

-- End to end working for snowpipe :=


# 3. API Integration (Azure)

## What It Does
Allows Snowflake to **call external HTTPS REST APIs** via External Functions. The API integration stores the Azure AD authentication details so Snowflake can securely invoke your Azure Function or API Management endpoint directly from a SQL query.

## Azure Components Involved
- **Azure Functions** (HTTP-triggered serverless compute) or **Azure API Management** (gateway/proxy)
- **Azure AD App Registration** (Service Principal that secures the API endpoint)
- Optionally: Azure API Management as a gateway in front of multiple backend services

## How It Works
1. You register an App in Azure AD and configure your Azure Function to require Azure AD tokens
2. You create an API Integration in Snowflake referencing the App's `Application ID` and `Tenant ID`
3. You create an External Function in Snowflake that maps to a specific endpoint URL
4. When SQL queries invoke the External Function, Snowflake acquires an OAuth token from Azure AD and calls your API

## When You Need It
- Calling an Azure Function from a SQL query (e.g., ML inference, geocoding, data enrichment)
- Integrating third-party REST APIs through Azure API Management as a proxy
- Running custom business logic that cannot be expressed in Snowflake SQL or stored procedures

## What It CANNOT Do
- Read/write files from Blob Storage (use Storage Integration instead)
- Listen for events or trigger automation (use Notification Integration instead)
- Handle SSO/OAuth login flows for users (use Security Integration instead)

In [ ]:
%%sql -r create_api_int
-- ============================================================
-- API INTEGRATION: Azure Functions
-- ============================================================

CREATE OR REPLACE API INTEGRATION azure_func_int
  API_PROVIDER = AZURE_API_MANAGEMENT
  AZURE_TENANT_ID = '<your-azure-tenant-id>'
  AZURE_AD_APPLICATION_ID = '<azure-ad-app-id>'  -- App Registration for your Azure Function
  API_ALLOWED_PREFIXES = (
    'https://my-function-app.azurewebsites.net/api/'
  )
  ENABLED = TRUE;

In [ ]:
%%sql -r external_func_example
-- ============================================================
-- EXTERNAL FUNCTION using the API Integration
-- ============================================================

-- CREATE OR REPLACE EXTERNAL FUNCTION sentiment_score(text VARCHAR)
--   RETURNS VARIANT
--   API_INTEGRATION = azure_func_int
--   AS 'https://my-function-app.azurewebsites.net/api/sentiment';

-- Usage:
-- SELECT product_name, sentiment_score(review_text)
-- FROM product_reviews;

## API Integration Flow

```
┌─────────────────────────────────────────────────────────────┐
│               API INTEGRATION FLOW                           │
│                                                             │
│  Snowflake SQL Query                                        │
│       │ SELECT sentiment_score(review_text) FROM ...        │
│       ▼                                                     │
│  External Function (resolves to API Integration)            │
│       │                                                     │
│       │ HTTPS POST with OAuth Bearer token (from Azure AD)  │
│       ▼                                                     │
│  Azure API Management / Azure Function                      │
│       │                                                     │
│       │ Validates token, processes request, returns JSON     │
│       ▼                                                     │
│  Result (VARIANT) returned to Snowflake query               │
└─────────────────────────────────────────────────────────────┘
```

### Azure Side Setup:
1. **Create an Azure Function App** with an HTTP trigger
2. **Register an App in Azure AD** — note the Application (client) ID
3. **Enable Authentication** on the Function App: require Azure AD tokens issued for your App Registration
4. **Create the API Integration** in Snowflake with the Application ID and Tenant ID
5. **Grant consent** — open the consent URL from `DESC INTEGRATION` and accept
6. **Create the External Function** in Snowflake, specifying the Function URL as the endpoint

### Limitations:
- External Functions use **synchronous** HTTP calls — long-running APIs may time out
- Each row in the query generates a request (batched into groups of up to 100 rows per HTTP call)
- The API must accept and return data in the [Snowflake External Function format](https://docs.snowflake.com/en/sql-reference/external-functions-data-format) (JSON array of `[row_number, result]` pairs)

# 4. Security Integration (Azure)

## What It Does
Manages **authentication and authorization** between Snowflake and Azure Active Directory (now called Microsoft Entra ID) for Single Sign-On, OAuth token validation, and automated user provisioning.

## Types of Security Integrations

| Sub-type | Purpose | Azure Feature | Direction |
|---|---|---|---|
| **SAML2** | Single Sign-On — users log in to Snowflake via Azure AD | Azure AD Enterprise App (SAML) | User → Azure AD → Snowflake |
| **External OAuth** | Validate OAuth tokens from external identity providers | Azure AD OAuth 2.0 | Tool → Azure AD token → Snowflake |
| **SCIM** | Automatically provision and deprovision users/groups from Azure AD to Snowflake | Azure AD SCIM client | Azure AD → Snowflake (sync) |

## When You Need It
- **SSO (SAML2)**: Users should log in to Snowflake using their corporate Azure AD credentials instead of a separate Snowflake password
- **External OAuth**: External tools (Power BI, Tableau, custom apps) authenticate to Snowflake using Azure AD-issued OAuth tokens
- **SCIM**: User accounts and role memberships should be automatically synced from Azure AD — no manual user creation in Snowflake

## Important Notes
- SAML2 and External OAuth are **different** integrations — SAML2 is for interactive browser-based login, while External OAuth is for programmatic/tool-based access
- SCIM requires a bearer token generated in Snowflake and configured in Azure AD's provisioning settings
- Only one SAML2 identity provider can be active per Snowflake account at a time

In [ ]:
%%sql -r create_security_int
-- ============================================================
-- SECURITY INTEGRATION: External OAuth (Azure AD)
-- Allows tools like Power BI to connect using Azure AD tokens
-- ============================================================

CREATE OR REPLACE SECURITY INTEGRATION azure_oauth_int
  TYPE = EXTERNAL_OAUTH
  ENABLED = TRUE
  EXTERNAL_OAUTH_TYPE = AZURE
  EXTERNAL_OAUTH_ISSUER = 'https://sts.windows.net/<your-tenant-id>/'
  EXTERNAL_OAUTH_JWS_KEYS_URL = 'https://login.microsoftonline.com/<your-tenant-id>/discovery/v2.0/keys'
  EXTERNAL_OAUTH_AUDIENCE_LIST = ('https://analysis.windows.net/powerbi/connector/Snowflake')
  EXTERNAL_OAUTH_TOKEN_USER_MAPPING_CLAIM = 'upn'
  EXTERNAL_OAUTH_SNOWFLAKE_USER_MAPPING_ATTRIBUTE = 'LOGIN_NAME';

# 5. Summary Comparison Table

| Feature | Storage Integration | Notification Integration | API Integration | Security Integration |
|---|---|---|---|---|
| **Direction** | Snowflake ↔ Blob (bidirectional file I/O) | Azure → Snowflake (inbound) or Snowflake → Azure (outbound) | Snowflake → External API (outbound only) | External → Snowflake (authentication) |
| **Azure Service** | Blob Storage / ADLS Gen2 | Event Grid + Storage Queue | Functions / API Management | Azure AD (Microsoft Entra ID) |
| **Auth Mechanism** | Service Principal + RBAC role on storage | Service Principal + Queue/Topic access | Azure AD App Registration + OAuth token | SAML2 / External OAuth / SCIM token |
| **Primary Use Case** | Load/unload data files via stages | Auto-ingest triggers, error alerts | Call REST APIs from SQL queries | SSO login, token-based auth, user sync |
| **SQL Object Created** | `STORAGE INTEGRATION` | `NOTIFICATION INTEGRATION` | `API INTEGRATION` | `SECURITY INTEGRATION` |
| **Required Privilege** | ACCOUNTADMIN or `CREATE INTEGRATION` on ACCOUNT | ACCOUNTADMIN or `CREATE INTEGRATION` on ACCOUNT | ACCOUNTADMIN or `CREATE INTEGRATION` on ACCOUNT | ACCOUNTADMIN or `CREATE INTEGRATION` on ACCOUNT |

## Key Takeaway

Each integration type maps to a **specific protocol and Azure service**. They are not interchangeable:
- **Storage Integration** uses Azure SDK-level blob access (not generic HTTP calls) — it is the only way to interact with files in stages
- **Notification Integration** uses queue-based event subscriptions (push/pull model) — it is the only way to trigger Snowpipe automatically
- **API Integration** only makes outbound HTTP POST/GET calls to a URL you specify — it cannot read blob files or listen for events
- **Security Integration** only handles authentication flows — it does not move data

**Use the right integration for the right job.**

# 6. Common Patterns (Azure)

## Pattern A: Full Data Pipeline (Storage + Notification)
Automatically loads new files into Snowflake as they arrive in blob storage.
```
Azure Blob ──(file lands)──► Event Grid ──► Storage Queue
                                                    │
                              Notification Integration (reads queue)
                                                    │
                                                    ▼
                              Snowpipe (AUTO_INGEST) ──► COPY INTO ──► Snowflake Table
```
**Integrations used:** Storage Integration (for the stage) + Notification Integration (for auto-ingest trigger)

## Pattern B: ML Enrichment Pipeline (Storage + API)
Loads raw data, enriches it via an external ML model, and exports results.
```
1. Load raw data:    Blob → Storage Integration → Stage → COPY INTO raw_table
2. Enrich via ML:    SELECT ml_function(col) FROM raw_table
                     (API Integration → Azure Function → ML model → returns prediction)
3. Export results:   COPY INTO @stage/export/ FROM enriched_table
                     (Storage Integration → writes back to Blob)
```
**Integrations used:** Storage Integration (load + unload) + API Integration (ML inference)

## Pattern C: Secure Dashboard Access (Security)
Allows BI tools to query Snowflake without storing passwords.
```
Power BI → requests OAuth token from Azure AD → presents token to Snowflake
         → Security Integration validates token → grants session with mapped user/role
```
**Integrations used:** Security Integration (External OAuth)

## Pattern D: End-to-End (Storage + Notification + API + Security)
```
1. Files land in blob            → Event Grid → Queue → Snowpipe loads data
2. Scheduled task enriches data  → External Function → Azure ML model
3. Analysts query via Power BI   → OAuth token → Security Integration → Snowflake
```

# Understanding Event Grid, Storage Queue, Inbound & Outbound

## The Two Azure Messaging Components

### Azure Event Grid — The Event Sensor
- **Role:** Detects that something happened (e.g., a file was uploaded to blob storage)
- **Behavior:** Fires immediately when an event occurs and pushes a notification to a subscriber
- **Does NOT store messages** — it is fire-and-forget; once delivered, the message is gone
- **Think of it as:** A motion detector that rings an alarm the instant it senses movement

### Azure Storage Queue — The Message Mailbox
- **Role:** Holds event messages durably until a consumer reads and deletes them
- **Behavior:** Messages sit in the queue for up to 7 days (configurable), waiting to be picked up
- **DOES store messages** — it is a buffer that decouples the producer from the consumer
- **Think of it as:** A mailbox that holds letters until you open and remove them

---

## Why Are Both Needed?

Event Grid **detects** events but cannot hold them. Storage Queue **holds** messages but cannot detect events on its own. Together they form a complete pipeline:

```
┌──────────────────────────────────────────────────────────────────────────┐
│                                                                          │
│   Azure Blob Storage                                                     │
│        │                                                                 │
│        │  (file is uploaded)                                             │
│        ▼                                                                 │
│   Event Grid  ─── "I detected a BlobCreated event"                       │
│        │                                                                 │
│        │  (pushes event message immediately)                             │
│        ▼                                                                 │
│   Storage Queue  ─── "I'll hold this message until someone reads it"     │
│        │                                                                 │
│        │  (Snowflake polls every ~1 minute)                              │
│        ▼                                                                 │
│   Snowflake Notification Integration  ─── "I see a new message"          │
│        │                                                                 │
│        │  (reads message, learns file path, deletes message from queue)  │
│        ▼                                                                 │
│   Snowpipe  ─── COPY INTO target_table FROM @stage/path/to/file.csv      │
│                                                                          │
└──────────────────────────────────────────────────────────────────────────┘
```

---

## Inbound vs Outbound (Always From Snowflake's Perspective)

| | **Inbound** | **Outbound** |
|---|---|---|
| **Direction** | Azure → Snowflake | Snowflake → Azure |
| **Meaning** | Snowflake **receives** a signal from outside | Snowflake **sends** a signal to outside |
| **Snowflake's role** | Consumer (reads/polls messages) | Producer (publishes messages) |
| **Azure service used** | Storage Queue (Snowflake polls it) | Event Grid Topic (Snowflake publishes to it) |
| **Provider in DDL** | `AZURE_STORAGE_QUEUE` | `AZURE_EVENT_GRID` |
| **Example use case** | Snowpipe auto-ingest ("a file arrived → load it") | Error alerts ("a task failed → notify ops team") |

### Why the naming makes sense:
- **Inbound** = information flows **in** to Snowflake (Snowflake is the receiver) [event is input for Snowflake]
- **Outbound** = information flows **out** of Snowflake (Snowflake is the sender) [event is output for Snowflake]

---

## Why Snowpipe Uses a Storage Queue (Not Event Grid Directly)

Snowpipe is a **consumer** of events — it needs to **read** messages at its own pace. Here is why a queue is required:

| Requirement | Event Grid | Storage Queue |
|---|---|---|
| Store messages until Snowflake is ready to read them | No — delivers once and forgets | Yes — holds until consumed |
| Allow Snowflake to poll on its own schedule (~1 min intervals) | No — push-only model | Yes — pull/poll model |
| Guarantee no message is lost if Snowflake is temporarily busy | No — if subscriber is unavailable, retries are limited | Yes — message persists in queue |
| Allow message to be deleted only after successful processing | No — no acknowledgment model | Yes — read → process → delete |

**Key insight:** Snowpipe does not generate events — it **consumes** them. A consumer needs a durable buffer (queue) to read from, not a real-time push channel (Event Grid). Event Grid's job ends once it delivers the message to the queue.

```
Event Grid = PRODUCER side (detects and pushes events) [Event is input for Snowflake]
Storage Queue = CONSUMER side (holds events for readers like Snowpipe) [Event is output for Snowflake]

Snowpipe is a consumer → it reads from the queue → therefore it needs AZURE_STORAGE_QUEUE
```

---

## Outbound Example: Snowflake Sending Alerts

When Snowflake is the **producer** (outbound), the roles reverse:

```
Snowflake detects error (e.g., task failure)
       │
       │  (publishes event)
       ▼
Event Grid Topic  ─── receives event from Snowflake
       │
       │  (routes to subscribers)
       ▼
Azure Logic App / Function / Email / Slack webhook
```

Here Snowflake uses `AZURE_EVENT_GRID` because it is **generating** an event (pushing), not consuming one.

---

## Summary Table

| Concept | Definition | Role in Snowpipe |
|---|---|---|
| **Event Grid** | Reactive event router — detects events and pushes them to subscribers | Detects "blob created" and pushes to the queue |
| **Storage Queue** | Durable message buffer — holds messages until a consumer reads them | Holds the event message until Snowflake polls it |
| **Inbound** | Snowflake receives information from Azure | Snowpipe reads from the queue (inbound flow) |
| **Outbound** | Snowflake sends information to Azure | Not used by Snowpipe — used for alerts/notifications |
| **Snowpipe** | Automated file loader that reacts to queue messages | Polls queue → reads file path → runs COPY INTO |

# 8. Permissions & Best Practices

## Who Can Create Integrations?
- Only **ACCOUNTADMIN** or a role with the `CREATE INTEGRATION` privilege on the account can create integrations
- After creation, grant `USAGE` on the integration to other roles so they can reference it in stages, pipes, or external functions

```sql
-- Grant usage to a role so they can create stages using this integration
GRANT USAGE ON INTEGRATION azure_blob_int TO ROLE data_engineer;
```

## Best Practices

1. **Least Privilege**: Use `STORAGE_ALLOWED_LOCATIONS` to restrict access to only the specific containers and paths needed
2. **Separate Integrations per Environment**: Create distinct integrations for dev, staging, and production to isolate access and simplify auditing
3. **Complete Consent Promptly**: The consent URL from `DESC INTEGRATION` expires — complete the Azure consent flow immediately after creation
4. **One Queue per Integration**: Do not share a single Azure Storage Queue across multiple notification integrations, as messages may be consumed by the wrong pipe
5. **Use STORAGE_BLOCKED_LOCATIONS**: Explicitly block sensitive paths (e.g., `/sensitive/`, `/keys/`) even if they fall outside your allowed locations, as a defense-in-depth measure
6. **Monitor with DESC**: Periodically run `DESC INTEGRATION <name>` to verify configuration, check consent status, and confirm the integration is `ENABLED = TRUE`
7. **Rotate Credentials**: If using SCIM, rotate the bearer token periodically. For Storage/Notification integrations, no credential rotation is needed since they use Azure AD Service Principals managed by Snowflake
8. **Document Your Integrations**: Maintain a record of which integrations map to which Azure resources, especially in multi-team environments